In [ ]:
pip install -U transformers accelerate torch

In [ ]:
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0)

In [ ]:
"""
HuggingFace GPT-2 Walkthrough (Readable Version)

What this script demonstrates:
1. Load a pretrained GPT-2 model
2. Convert raw text into tokens
3. Run a causal (decoder-only) forward pass
4. Inspect:
   - next-token probability distributions (logits)
   - sequence-level loss
   - per-token negative log-likelihood (NLL)

The goal is to make the mechanics of a decoder-only LM explicit.
"""

# =========================
# Imports
# =========================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# Configuration
# =========================

# HuggingFace model identifier
MODEL_NAME = "gpt2"  # small GPT-2 (~124M parameters)

# Input prompt to analyze
INPUT_TEXT = (
    "Hello from VS Code on a Tesla T4! "
    "Let's inspect GPT-2 logits and loss."
)

# Decide whether to run on GPU or CPU
# (Tesla T4 will appear as CUDA if configured correctly)
compute_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running on device: {compute_device}")
if compute_device == "cuda":
    print("GPU model:", torch.cuda.get_device_name(0))


# =========================
# Load Tokenizer
# =========================

# The tokenizer converts raw text -> token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 does not define a padding token by default.
# We reuse the end-of-sequence token for padding to keep batching safe.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# =========================
# Load Model
# =========================

# AutoModelForCausalLM loads:
# - GPT-2 transformer decoder
# - Language modeling head (linear layer over vocab)
language_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move all model parameters to the selected device
language_model.to(compute_device)

# Switch to evaluation mode:
# - disables dropout
# - ensures deterministic behavior
language_model.eval()


# =========================
# Tokenize Input Text
# =========================

# Convert text into model-ready tensors
tokenized_inputs = tokenizer(
    INPUT_TEXT,
    return_tensors="pt",   # return PyTorch tensors
    padding=True,          # pad to batch max length
    truncation=True,       # truncate if text is too long
)

# Extract tensors and move them to the same device as the model
input_token_ids = tokenized_inputs["input_ids"].to(compute_device)
attention_mask = tokenized_inputs["attention_mask"].to(compute_device)

print("\n--- Tokenization ---")
print("Original text:")
print(INPUT_TEXT)
print("\nToken tensor shape:", tuple(input_token_ids.shape))
print("Attention mask shape:", tuple(attention_mask.shape))
print("\nDecoded tokens (sanity check):")
print(tokenizer.decode(input_token_ids[0]))


# =========================
# Forward Pass (No Gradients)
# =========================

# We disable gradient tracking because:
# - we are not training
# - it saves memory and computation
#
# Passing labels=input_token_ids tells HuggingFace to:
# - shift tokens internally
# - compute causal LM loss (next-token prediction)
with torch.no_grad():
    model_outputs = language_model(
        input_ids=input_token_ids,
        attention_mask=attention_mask,
        labels=input_token_ids,
    )

# Extract outputs
logits = model_outputs.logits  # raw scores before softmax
sequence_loss = model_outputs.loss  # mean cross-entropy loss

print("\n--- Forward Pass Outputs ---")
print("Logits shape:", tuple(logits.shape))
print("Sequence loss:", float(sequence_loss))


# =========================
# Inspect Next-Token Predictions
# =========================

# We inspect predictions at a specific token position
# (chosen deliberately in the *middle* of the sequence)
batch_index = 0
token_position = min(5, logits.shape[1] - 1)

# Convert logits -> probabilities for that position
next_token_probabilities = torch.softmax(
    logits[batch_index, token_position],
    dim=-1
)

# Extract top-K most likely next tokens
TOP_K = 10
topk_result = torch.topk(next_token_probabilities, TOP_K)

top_token_ids = topk_result.indices.tolist()
top_token_probs = topk_result.values.tolist()

# Decode token IDs back to text fragments
top_token_strings = [
    tokenizer.decode([token_id]) for token_id in top_token_ids
]

print(f"\n--- Top-{TOP_K} predictions at position {token_position} ---")
for token_str, prob in zip(top_token_strings, top_token_probs):
    print(f"{repr(token_str):>12}  p={prob:.4f}")


# =========================
# Manual Per-Token Loss Inspection
# =========================

# GPT-style language modeling predicts:
# token[t+1] given tokens[0:t]
#
# Therefore:
# - final token has no target
# - logits and labels must be shifted
shifted_logits = logits[:, :-1, :].contiguous()
shifted_labels = input_token_ids[:, 1:].contiguous()

# Convert logits to log-probabilities
log_probabilities = torch.log_softmax(shifted_logits, dim=-1)

# Gather log-probabilities assigned to the *true* next tokens
negative_log_likelihood = -log_probabilities.gather(
    dim=-1,
    index=shifted_labels.unsqueeze(-1)
).squeeze(-1)  # shape: [batch_size, seq_len - 1]

print("\n--- Per-token NLL (first sequence) ---")

# Move to CPU for readable printing
nll_values = negative_log_likelihood[0].detach().float().cpu()

for position, nll_value in enumerate(nll_values[:12]):
    current_token = tokenizer.decode([input_token_ids[0, position].item()])
    next_token = tokenizer.decode([input_token_ids[0, position + 1].item()])
    print(
        f"pos {position:02d}: "
        f"{repr(current_token)} -> {repr(next_token)} | "
        f"NLL={nll_value:.3f}"
    )

print("\nDone.")

Device: cuda
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


--- Tokenization ---
Text: Hello from VS Code on a Tesla T4! Let's inspect GPT-2 logits and loss.
input_ids shape: (1, 22)
attention_mask shape: (1, 22)
Decoded back: Hello from VS Code on a Tesla T4! Let's inspect GPT-2 logits and loss.


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



--- Forward Pass Outputs ---
logits shape: (1, 22, 50257)
loss: 6.055650234222412

--- Top-10 predictions at position 5 ---
   ' recent'  p=0.0868
 ' personal'  p=0.0263
      ' new'  p=0.0178
     ' whim'  p=0.0141
     ' very'  p=0.0123
      ' few'  p=0.0122
' different'  p=0.0107
    ' daily'  p=0.0099
   ' couple'  p=0.0099
  ' Windows'  p=0.0084

--- Per-token NLL (first example) ---
pos 00: 'Hello' -> ' from' | NLL=5.180
pos 01: ' from' -> ' VS' | NLL=10.929
pos 02: ' VS' -> ' Code' | NLL=2.834
pos 03: ' Code' -> ' on' | NLL=4.966
pos 04: ' on' -> ' a' | NLL=4.765
pos 05: ' a' -> ' Tesla' | NLL=9.045
pos 06: ' Tesla' -> ' T' | NLL=4.725
pos 07: ' T' -> '4' | NLL=5.509
pos 08: '4' -> '!' | NLL=5.190
pos 09: '!' -> ' Let' | NLL=5.143
pos 10: ' Let' -> "'s" | NLL=0.392
pos 11: "'s" -> ' inspect' | NLL=10.342

Done.
